In [ ]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192")

store = {}

# InMemoryChatMessageHistory would store messages history as user and AI for tempory, you can replace it with any other implementation of ChatMessageHistory to store the history in a database or file system.
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"), # this part injects the history of convo from RunnableWithMessageHistory into the prompt, so the model can have the context of previous messages in the conversation.
    ("human", "{input}")
])

chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history, 
    input_messages_key="input",
    history_messages_key="history"
)

while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    res = chain_with_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": "user1"}}
    )
    
    print("AI:", res.content)

In [ ]:
## Stateless RAG

from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

# 1. Load LLM (Groq)
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

# 2. Load Documents
loader = TextLoader("data.txt")  # your file
documents = loader.load()

# 3. Split Documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.split_documents(documents)

# 4. Convert to Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 5. Store in Vector DB (FAISS)
vectorstore = FAISS.from_documents(docs, embeddings)

# 6. Retriever
retriever = vectorstore.as_retriever()

# 7. Prompt Template (IMPORTANT 🔥)
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer ONLY from the provided context.
If the answer is not in context, say "I don't know".

Context:
{context}

Question:
{question}
""")

# 8. RAG Pipeline (NO MEMORY)
def rag_pipeline(question):
    # Retrieve docs
    retrieved_docs = retriever.get_relevant_documents(question)
    
    # Combine context
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Create prompt
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    
    # LLM call
    response = llm.invoke(final_prompt)
    
    return response.content

# 9. Run
while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    answer = rag_pipeline(q)
    
    print("AI:", answer)

In [1]:
print("dh")

dh


In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import MessagesPlaceholder,ChatMessagePromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

llm = ChatGroq(model="llama3-8b-8192", temperature=0)

loader = TextLoader('docs.txt')
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
doc = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(doc,embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

chat_prompt = ChatPromptTemplate(
    ("System","Answer from the given context")
    MessagesPlaceholder(variable_name="history")
    ("human","context: {context}\n\n, Question: {Question}\n\n")
)

store={}

def get_session_id(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    
    return store[session_id]

def retrieve(inputs):
    question = inputs["Question"]
    docs = retriever._get_relevant_documents(question)
    content = "\n\n".join(doc.page_document for doc in doc)
    return {"context":{content} ,"Question":{question}}

chain = retriever|prompt|llm

chat = RunnableWithMessageHistory(
    chain,
    get_session_history,
    history_messages_key="history",
    input_messages_key="input"
)


    





In [ ]:
# Statefull RAG

from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. LLM
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

# 2. Load Docs
loader = TextLoader("data.txt")
documents = loader.load()

# 3. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)

# 4. Embeddings + Vector DB
embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

# 5. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # it use cosine similarity by default, you can change it to other similarity metrics supported by FAISS if needed.

# 6. Prompt (IMPORTANT 🔥)
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful AI assistant. Answer using the provided context. "
     "If not found, say 'I don't know'."),
    
    MessagesPlaceholder(variable_name="history"),
    
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

# 7. RAG Function inside LCEL
def retrieve_context(inputs):
    question = inputs["question"]
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    return {"context": context, "question": question}

# 8. Chain
chain = (
    retrieve_context
    | prompt
    | llm
)

# 9. Memory Store (multi-user)
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# 10. Add Memory
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

# 11. Run
while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    res = chain_with_memory.invoke(
        {"question": q},
        config={"configurable": {"session_id": "user1"}}
    )
    
    print("AI:", res.content)

In [ ]:
# =========================================================
# 1. ENV
# =========================================================

from dotenv import load_dotenv
load_dotenv()

# =========================================================
# 2. LLM
# =========================================================

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

# =========================================================
# 3. DOCUMENT LOADING
# =========================================================

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data.txt")
documents = loader.load()

# =========================================================
# 4. CHUNKING
# =========================================================

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

# =========================================================
# 5. EMBEDDING MODEL
# =========================================================

from langchain.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en",
    encode_kwargs={
        "normalize_embeddings": True # better for cosine similarity calculation
    }
)

# =========================================================
# 6. VECTOR DB (QDRANT + HNSW) (by ANN)
# =========================================================

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from langchain_qdrant import QdrantVectorStore

client = QdrantClient(
    url="http://localhost:6333" 
)

# Create collection
client.recreate_collection(
    collection_name="rag_collection",

    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

# Store documents
vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
    collection_name="rag_collection"
)

# =========================================================
# 7. RETRIEVER (MMR FOR CONTEXT PACKING)
# =========================================================

retriever = vectorstore.as_retriever(
    search_type="mmr", # context packing strategy pick only top non redudant data

    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

# =========================================================
# 8. QUERY EXPANSION (Multi query generation)
# =========================================================

# from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

from langchain_core.prompts import ChatPromptTemplate

query_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate 3 different versions of the user's question.

Original question:
{question}

Return only the 3 alternative questions, one per line.""")
])



llm_chain = LLMChain(
    llm=llm,
    prompt=query_prompt
)

multi_query_retriever = MultiQueryRetriever(
    retriever=retriever,
    llm_chain=llm_chain
)

# =========================================================
# 9. RERANKER
# =========================================================

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)


# =========================================================
# 10. RETRIEVAL + RERANKING PIPELINE
# =========================================================

def retrieve_context(inputs):

    question = inputs["question"]

    # ---------------------------------------------
    # QUERY EXPANSION + RETRIEVAL
    # ---------------------------------------------

    retrieved_docs = multi_query_retriever.invoke(question)

    # ---------------------------------------------
    # RERANKING
    # ---------------------------------------------

    pairs = [
        [question, doc.page_content]
        for doc in retrieved_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # ---------------------------------------------
    # CONTEXT PACKING
    # ---------------------------------------------

    top_docs = [doc for doc, _ in ranked_docs[:3]]

    context = "\n\n".join([
        doc.page_content
        for doc in top_docs
    ])

    return {
        "context": context,
        "question": question
    }

# =========================================================
# 11. PROMPT
# =========================================================

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

prompt = ChatPromptTemplate.from_messages([

    (
        "system",

        "You are a helpful AI assistant.\n"
        "Answer ONLY from the provided context.\n"
        "If answer is not found, say 'I don't know'."
    ),

    MessagesPlaceholder(variable_name="history"),

    (
        "human",

        "Context:\n{context}\n\n"
        "Question:\n{question}"
    )
])



# =========================================================
# 12. CHAIN
# =========================================================

chain = (
    retrieve_context
    | prompt
    | llm
)

# =========================================================
# 13. MEMORY (KEEPING YOUR SAME LOGIC)
# =========================================================

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,

    input_messages_key="question",
    history_messages_key="history"
)

# =========================================================
# 14. CHAT LOOP
# =========================================================

while True:

    q = input("You: ")

    if q.lower() in ["exit", "quit"]:
        break

    res = chain_with_memory.invoke(

        {"question": q},

        config={
            "configurable": {
                "session_id": "user1"
            }
        }
    )

    print("\nAI:", res.content)

In [7]:
import httpx
import os
from dotenv import load_dotenv  

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

async def call_groq(prompt):
    
    async with httpx.AsyncClient(timeout=30) as client:
        response= await client.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {GROQ_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "openai/gpt-oss-120b",
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            }
        )
        
        if response.status_code == 200:
            
            data = response.json()
            return data["choices"][0]["message"]["content"]
        
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
        
    
result = await call_groq("Hello, how are you?")
print(result)

Hello! I'm doing great, thank you for asking. How can I help you today?


In [11]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

async def call_llm(prompt,provider="ollama"):
    
    if provider == 'groq':
        
        base_url= "https://api.groq.com/openai/v1"
        model = "openai/gpt-oss-120b"
        
        headers={
            "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
            "Content-Type": "application/json"
        }
        
    else:
        
        base_url= "https://localhost:11434/v1",
        model= "mistral"
        
        headers={
            "Content-Type": "application/json"
        }
        
    async with httpx.AsyncClient(timeout=30) as client:
            
            response= await client.post(
                f"{base_url}/chat/completions",
                headers=headers,
                json={
                    "model": model,
                    "messages":[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]
                }
            )

            if response.status_code == 200:
                data = response.json()
                return data["choices"][0]["message"]["content"]
            
            else:
                raise Exception(f"Error: {response.status_code}, {response.text}")
            
            

result = await call_llm("Hello, how are you?", provider="groq")
print(result)
        
    

Hello! I'm doing well, thank you for asking. How can I assist you today?


In [ ]:
import os 
import httpx
from dotenv import load_dotenv
load_dotenv()

async def call_llm(prompt,provider="groq"):
    
    if provider == "groq":
        base_url="https://api.groq.com//openai/v1"
        model= "openai/gpt-oss-120b"
        
        headers={
            "Authorization": f"Bearer {os.getenv("GORQ_API_KEY")}",
            "Content-Type": "application/json"
        }

    else:
        base_url="localhost/112437/v1"
        model="mistral"
        
        headers={
            "Content-Type":"application/json"
        }
        
    async with httpx.AsyncClient(timeout=30) as client:
        
        response= await client.post(
            f"{base_url}/chat/compilations",
            headers=headers,
            json={
                "model":model,
                "message":[
                    {
                        'role':'User',
                        'content':prompt
                    }
                ]
            }
        )
        
        if response.status_code == 200:
            data = response.json()
            return data['choices'][0]['message']['content']
        
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
        

In [ ]:
import asyncio

queue = asyncio.Queue() # It creates a asynchronous queue created in the memory

async def producer():
    
    for i in range(5):
        
        await queue.put(i)
        
        print(f"Produced: {i}")

async def consumer():
    
    while True:
        
        item = await queue.get()
        
        print(f"Consumed: {item}")
        
        await asyncio.sleep(1)
        
        queue.task_done() # This would notify the queue that the item has been processed, so it can be removed from the queue.
        
async def main():
    consumer_task= asyncio.create_task(consumer()) # This would run the consumer in the background, so it can consume items from the queue while the producer is producing them.
    
    await producer() # Here producer would start add the jobs to the queue, and the consumer would start consuming them in the background.
    
    print("Producer finished producing items.")
    
    await queue.join() # This would wait until all the items in queue have been consumed, and the consumer has called task_done() for each item.
    
    print("All items have been consumed.")
    
    consumer_task.cancel() # Thiis would stop the consumer once all the jobs consumed to prevent it from running forever, since the consumer is in an infinite loop.
    
asyncio.run(main())

## Queue Processing

#### Producer

have the await queue.put()

#### Consumer

have the await queue.get()

queue.task_done()

#### Main

await asyncio.create_task(consumer())

await queue.join()

consumer.cancel()


In [ ]:
import asyncio
import uuid

queue = asyncio.Queue() 

jobs={}

async def create_job(resume_data):
    
    job_id= str(uuid.uuid4())
    
    jobs[job_id]={
        "resume_data": resume_data,
        "status": "pending"
    }
    
    job={
        "job_id": job_id,
        "resume_data": resume_data,
    }
    
    await queue.put(job)
    
    print(f"Job {job_id} created and added to the queue.")
    
    return job_id

async def worker():
    
    while True:
        
        job= await queue.get()
        
        job_id= job["job_id"]
        resume_data= job["resume_data"]
        
        jobs[job_id]["status"]= "in_progress"
        
        print(f"Processing job {job_id} with resume data: {resume_data}")
        
        # Simulate OCR
        print("  → Extracting text...")
        await asyncio.sleep(2)

        # Simulate LLM
        print("  → Calling LLM...")
        await asyncio.sleep(3)

        # Simulate database
        print("  → Saving result...")
        await asyncio.sleep(1)
        
        jobs[job_id]["status"]= "completed"
        
        print(f"Job {job_id} completed.")
        
        queue.task_done()
        
async def main():
    
    worker_task= asyncio.create_task(worker())
    
    job1 = await create_job("Resume data for John Doe")
    job2 = await create_job("Resume data for Jane Smith")
    
    print(f"Created jobs: {job1}, {job2}")
    
    await queue.join() # Wait until all jobs are processed
    
    print("All jobs have been processed.")
    
    worker_task.cancel() # Stop the worker after all jobs are done
    

asyncio.run(main())